# Complete Guide to Normalization in PyTorch: Batch vs Layer Normalization

Normalization is one of the most important techniques in modern deep learning. It stabilizes the training process, mitigates exploding and vanishing gradients, and allows the model to converge much faster using larger learning rates.

In this guide, we'll go deep into the differences between **Batch Normalization** and **Layer Normalization**, how to use them with **1D, 2D, and 3D** data, and how to correctly set the **`num_features`** parameter.

---

## 1. Layer Normalization vs. Batch Normalization

The key difference between the two methods is **which dimension** the mean and variance are computed over to perform the normalization.

#### Batch Normalization (BatchNorm)
* **How it works:** Computes mean and variance along the **batch dimension** for each channel individually.
* **Dependency:** Heavily dependent on batch size. With very small batches (e.g. 2 or 4), the mean and variance estimates are noisy, hurting training.
* **Train vs. Eval behavior:** During training, it computes batch statistics and updates a running mean/variance. In eval mode (`model.eval()`), it freezes those running statistics to normalize test data.
* **Ideal use case:** Convolutional Neural Networks (CNNs) for Computer Vision.
* `nn.BatchNormXd(num_fet = x)`

#### Layer Normalization (LayerNorm)
* **How it works:** Computes mean and variance along the **feature dimensions** for each example independently.
* **Dependency:** Completely independent of batch size. The computation for one example in a batch of 32 is identical to a batch of 1.
* **Train vs. Eval behavior:** Identical behavior in both `model.train()` and `model.eval()`. No running statistics stored.
* **Ideal use case:** NLP, RNNs, and **Transformer**-based architectures (BERT, GPT, etc.).
* `nn.LayerNorm(num_fet = x)`

---

## 2. Dimensions in PyTorch: 1D, 2D, 3D

PyTorch provides normalization class variants to match the tensor format each layer type expects.

### 1D Data (`BatchNorm1d`) (2d - 3d)
* **Common use:** Fully Connected networks (MLPs) or simple temporal sequences.
* **Accepted tensor formats:**
  * `(N, C)` where $N$ is the batch and $C$ is the number of features/channels.
  * `(N, C, L)` where $L$ is the sequence length.

### 2D Data (`BatchNorm2d`) (3d+)
* **Common use:** Images or feature maps from 2D convolutions.
* **Accepted tensor format:**
  * `(N, C, H, W)` where $H$ is Height and $W$ is Width.

### 3D Data (`BatchNorm3d`) (4d+)
* **Common use:** Videos (with a time/frames dimension) or 3D medical images (CT scans, MRIs).
* **Accepted tensor format:**
  * `(N, C, D, H, W)` where $D$ is Depth/Time.

> **What about LayerNorm?** `nn.LayerNorm` is more flexible and has no variants like `LayerNorm2d`. You pass it the shape of the last dimensions you want to normalize (see below).

---

## 3. The `num_features` Parameter and Configurations

Configuration differs slightly between techniques. Here's how to instantiate each one.

### Configuring `nn.BatchNormXd`
For Batch Normalization, the required parameter is **`num_features`**. It must always receive the **number of channels (`C`)** of the incoming tensor.


## 4. Which Dimension Does Each One Normalize?

A simple rule to always remember:

- BatchNorm  →  always the second dim (C)
- LayerNorm  →  always the last dim(s)

In [6]:
import torch
import torch.nn as nn


input = torch.rand(5, 5)
print(f'Input before: {input}')

normb = nn.BatchNorm1d(input.shape[1])
norml = nn.LayerNorm(input.shape[1])


input_normb = normb(input)
input_norml = norml(input)

Input before: tensor([[0.1566, 0.3632, 0.8921, 0.0211, 0.4806],
        [0.5803, 0.3747, 0.8705, 0.3216, 0.0491],
        [0.1827, 0.2015, 0.0012, 0.5023, 0.3959],
        [0.4716, 0.3021, 0.0269, 0.9785, 0.4499],
        [0.3433, 0.4254, 0.2701, 0.0880, 0.7664]])


In [7]:
print(f'Input after the BatchNorm: {input_normb}')

Input after the BatchNorm: tensor([[-1.1656,  0.3878,  1.2167, -1.0508,  0.2280],
        [ 1.4296,  0.5377,  1.1620, -0.1767, -1.6556],
        [-1.0060, -1.7174, -1.0418,  0.3492, -0.1417],
        [ 0.7637, -0.4068, -0.9767,  1.7346,  0.0937],
        [-0.0218,  1.1987, -0.3602, -0.8563,  1.4756]],
       grad_fn=<NativeBatchNormBackward0>)


In [ ]:
print(f'Input after the LayerNorm: {input_norml}')

Input after the LayerNorm: tensor([[-0.7525, -0.0652,  1.6954, -1.2035,  0.3258],
        [ 0.5142, -0.2354,  1.5724, -0.4289, -1.4223],
        [-0.4226, -0.3151, -1.4580,  1.4014,  0.7942],
        [ 0.0831, -0.4634, -1.3511,  1.7182,  0.0131],
        [-0.1579,  0.2090, -0.4854, -1.2995,  1.7338]],
       grad_fn=<NativeLayerNormBackward0>)


: 